In [3]:
from pyspark.sql.functions import when, col, to_timestamp, lit, current_timestamp
from pyspark.sql.types import TimestampType

# Get Lakehouse path directly — bypasses all namespace issues
lh = notebookutils.lakehouse.get("lh_northwind_erp")
LH_PATH = lh.properties["abfsPath"]

print(f"Lakehouse path: {LH_PATH}")

StatementMeta(, 7d5469cb-6e78-433e-91e5-264482acc1b7, 5, Finished, Available, Finished, False)

Lakehouse path: abfss://42e264c9-ac1b-48f9-975c-2bbaf3322f18@onelake.dfs.fabric.microsoft.com/968d0d78-056b-4db2-a68d-e3d193fb22af


In [6]:
# Read Bronze orders directly from Delta path
df_orders = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/orders")

df_silver_orders = df_orders \
    .withColumn("shipRegion",
        when(col("shipRegion") == "NULL", None)
        .otherwise(col("shipRegion"))) \
    .withColumn("shippedDate",
        to_timestamp(
            when(col("shippedDate") == "NULL", None)
            .otherwise(col("shippedDate")),
            "yyyy-MM-dd HH:mm:ss.SSS")) \
    .withColumn("_silver_processed_at", current_timestamp())

bronze_count = df_orders.count()
silver_count = df_silver_orders.count()

if bronze_count != silver_count:
    raise ValueError(f"Row mismatch! Bronze={bronze_count}, Silver={silver_count}")

df_silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/silver_orders")

print(f"[silver_orders] {silver_count} rows written successfully")

StatementMeta(, 9459fd94-6d5f-4cfd-ac05-78a1cff6ec4d, 8, Finished, Available, Finished, False)

[silver_orders] 830 rows written successfully


In [4]:
# Cell 3 — Silver customers: fix string NULLs

df_customers = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/customers")

df_silver_customers = df_customers \
    .withColumn("region",
        when(col("region") == "NULL", None)
        .otherwise(col("region"))) \
    .withColumn("fax",
        when(col("fax") == "NULL", None)
        .otherwise(col("fax"))) \
    .withColumn("_silver_processed_at", current_timestamp())

bronze_count = df_customers.count()
silver_count = df_silver_customers.count()

if bronze_count != silver_count:
    raise ValueError(f"Row mismatch! Bronze={bronze_count}, Silver={silver_count}")

df_silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/silver_customers")

print(f"[silver_customers] {silver_count} rows written successfully")

StatementMeta(, 7d5469cb-6e78-433e-91e5-264482acc1b7, 6, Finished, Available, Finished, False)

[silver_customers] 91 rows written successfully


In [5]:
# Cell 4 — Silver suppliers: fix string NULLs

df_suppliers = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/suppliers")

df_silver_suppliers = df_suppliers \
    .withColumn("region",
        when(col("region") == "NULL", None)
        .otherwise(col("region"))) \
    .withColumn("fax",
        when(col("fax") == "NULL", None)
        .otherwise(col("fax"))) \
    .withColumn("_silver_processed_at", current_timestamp())

bronze_count = df_suppliers.count()
silver_count = df_silver_suppliers.count()

if bronze_count != silver_count:
    raise ValueError(f"Row mismatch! Bronze={bronze_count}, Silver={silver_count}")

df_silver_suppliers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/silver_suppliers")

print(f"[silver_suppliers] {silver_count} rows written successfully")

StatementMeta(, 7d5469cb-6e78-433e-91e5-264482acc1b7, 7, Finished, Available, Finished, False)

[silver_suppliers] 29 rows written successfully


In [6]:
# Cell 5 — Silver employees: fix string NULLs

df_employees = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/employees")

df_silver_employees = df_employees \
    .withColumn("region",
        when(col("region") == "NULL", None)
        .otherwise(col("region"))) \
    .withColumn("reportsTo",
        when(col("reportsTo") == "NULL", None)
        .otherwise(col("reportsTo"))) \
    .withColumn("_silver_processed_at", current_timestamp())

bronze_count = df_employees.count()
silver_count = df_silver_employees.count()

if bronze_count != silver_count:
    raise ValueError(f"Row mismatch! Bronze={bronze_count}, Silver={silver_count}")

df_silver_employees.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/silver_employees")

print(f"[silver_employees] {silver_count} rows written successfully")

StatementMeta(, 7d5469cb-6e78-433e-91e5-264482acc1b7, 8, Finished, Available, Finished, False)

[silver_employees] 9 rows written successfully


In [7]:
# Cell 6 — Pass-through tables (no issues found in profiling)

passthrough_tables = ["products", "categories", "shippers", "order_details"]

for table in passthrough_tables:
    df = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/{table}")
    
    df_silver = df.withColumn("_silver_processed_at", current_timestamp())
    
    bronze_count = df.count()
    silver_count = df_silver.count()
    
    if bronze_count != silver_count:
        raise ValueError(f"[silver_{table}] Row mismatch! Bronze={bronze_count}, Silver={silver_count}")
    
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{LH_PATH}/Tables/dbo/silver_{table}")
    
    print(f"[silver_{table}] {silver_count} rows written successfully")

StatementMeta(, 7d5469cb-6e78-433e-91e5-264482acc1b7, 9, Finished, Available, Finished, False)

[silver_products] 77 rows written successfully
[silver_categories] 8 rows written successfully
[silver_shippers] 3 rows written successfully
[silver_order_details] 2155 rows written successfully
